# Valoricore × LlamaIndex

**Deterministic vector memory with cryptographic audit trails — natively in LlamaIndex.**

This notebook shows:
1. One-line setup — `ValoricoreLlamaIndex` replaces Chroma / Weaviate / Pinecone
2. Building a `VectorStoreIndex` from documents
3. Querying with `as_query_engine()`
4. Low-level node add / query / delete
5. Cryptographic state proof — BLAKE3 hash
6. Snapshot & bit-exact restore

No server required. Everything runs locally in this Colab cell.

[![GitHub](https://img.shields.io/badge/GitHub-Valori--Kernel-6c47ff?logo=github)](https://github.com/varshith-Git/Valori-Kernel)

## 1 · Install

In [ ]:
%%capture
!pip install "valoricore[llamaindex,local]"
# llamaindex → llama-index-core
# local      → sentence-transformers (offline embeddings, no API key)

## 2 · Imports

In [ ]:
from valoricore.integrations import ValoricoreLlamaIndex

from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.core.schema import TextNode, Document
from llama_index.core.vector_stores.types import VectorStoreQuery

print("Imports OK ✓")

## 3 · Configure a local embedding model

We use `sentence-transformers` so the whole notebook runs offline.
Swap with `OpenAIEmbedding()` or any LlamaIndex-compatible embedder.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Tell LlamaIndex to use this embedder globally
Settings.embed_model = embed_model
Settings.llm         = None   # no LLM needed for pure retrieval

dim = len(embed_model.get_text_embedding("test"))
print(f"Embedding dim: {dim}")

## 4 · One-line initialization

`ValoricoreLlamaIndex` is a drop-in `BasePydanticVectorStore`. No adapter wiring, no HTTP server.

In [ ]:
vector_store = ValoricoreLlamaIndex(
    path       = "/tmp/valori_llamaindex_demo",
    index_kind = "hnsw",   # "bruteforce" | "hnsw" | "ivf"
)

# To connect to a remote node:
#   vector_store = ValoricoreLlamaIndex(remote="http://my-node:3000")

print("Vector store initialized ✓")
print(f"stores_text      : {vector_store.stores_text}")
print(f"is_embedding_query: {vector_store.is_embedding_query}")

## 5 · Build a VectorStoreIndex from documents

This is the standard LlamaIndex RAG workflow — plug Valoricore in place of any other vector store.

In [ ]:
documents = [
    Document(text="Valoricore is a deterministic vector database built in Rust."),
    Document(text="Q16.16 fixed-point arithmetic produces identical results on x86, ARM, and RISC-V."),
    Document(text="Every insert is backed by a BLAKE3 Merkle proof."),
    Document(text="The event log is append-only and fsynced before any live state change."),
    Document(text="Crash recovery is mathematically proven, not just claimed."),
    Document(text="Leader-follower replication uses tokio::sync::watch for race-free coordination."),
    Document(text="LlamaIndex users can swap Chroma for Valoricore with one import."),
    Document(text="The HNSW index enables sub-millisecond search at millions of records."),
]

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context = storage_context,
    show_progress   = True,
)

print(f"\nIndex built. State hash: {vector_store.get_state_hash()[:16]}...")

## 6 · Query with as_query_engine()

Requires an LLM. Below we use a commented-out OpenAI block and also show retriever-only mode (no LLM needed).

In [ ]:
# ── Retriever-only (no LLM required) ─────────────────────────────────────────
retriever = index.as_retriever(similarity_top_k=3)
nodes     = retriever.retrieve("How does crash recovery work?")

print(f"Retrieved {len(nodes)} nodes:\n")
for i, n in enumerate(nodes, 1):
    print(f"{i}. score={n.score:.4f}  →  {n.text}")

In [ ]:
# ── Full RAG with OpenAI (requires OPENAI_API_KEY) ───────────────────────────
# Uncomment to run:

# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."
#
# from llama_index.llms.openai import OpenAI
# from llama_index.embeddings.openai import OpenAIEmbedding
#
# Settings.llm         = OpenAI(model="gpt-4o-mini")
# Settings.embed_model = OpenAIEmbedding()
#
# openai_store = ValoricoreLlamaIndex(path="/tmp/valori_openai")
# ctx          = StorageContext.from_defaults(vector_store=openai_store)
# idx          = VectorStoreIndex.from_documents(documents, storage_context=ctx)
#
# engine   = idx.as_query_engine(similarity_top_k=4)
# response = engine.query("How does Valoricore prove crash recovery?")
# print(response)

## 7 · Low-level node add / query

Useful when you already have pre-computed embeddings and want full control.

In [ ]:
# Build a fresh store so we can inspect the low-level API clearly
raw_store = ValoricoreLlamaIndex(path="/tmp/valori_raw")

# Create nodes with pre-computed embeddings
node_texts = [
    "IVF index uses deterministic k-means with FNV-1a seeding.",
    "BLAKE3 is a cryptographic hash function with parallel tree hashing.",
    "Q16.16 format: 16 integer bits + 16 fractional bits stored as i32.",
]

nodes = [
    TextNode(
        text      = t,
        id_       = f"raw-node-{i}",
        embedding = embed_model.get_text_embedding(t),
        metadata  = {"index": i},
    )
    for i, t in enumerate(node_texts)
]

inserted_ids = raw_store.add(nodes)
print(f"Inserted node IDs: {inserted_ids}")

In [ ]:
# Low-level query — pass a pre-computed embedding
q = VectorStoreQuery(
    query_embedding = embed_model.get_text_embedding("BLAKE3 cryptographic hash"),
    similarity_top_k = 2,
)

result = raw_store.query(q)

print(f"\nTop {len(result.nodes)} results:")
for node, sim, nid in zip(result.nodes, result.similarities, result.ids):
    print(f"  id={nid}  sim={sim:.4f}  →  {node.text}")

## 8 · Score semantics

Valoricore returns similarity in **(0, 1]** via `1/(1+L2²)`. A score of **1.0 = perfect match**.
This is compatible with LlamaIndex's `NodeWithScore` expectation.

In [ ]:
# Verify score range
for node, sim in zip(result.nodes, result.similarities):
    assert 0.0 < sim <= 1.0, f"Score out of range: {sim}"
    print(f"  score={sim:.6f}  (valid (0,1] range) ✓")

## 9 · Delete nodes

In [ ]:
h_before = raw_store.get_state_hash()
print(f"Hash before delete: {h_before[:24]}...")

raw_store.delete("raw-node-1")

h_after = raw_store.get_state_hash()
print(f"Hash after delete : {h_after[:24]}...")
print(f"Hash changed: {h_before != h_after}")

## 10 · Cryptographic state proof

The 64-char BLAKE3 root is **deterministic across all machines and OS versions**.
Identical insertions → identical hash. Use it to prove dataset integrity.

In [ ]:
h1 = vector_store.get_state_hash()
print(f"Before new insert: {h1}")

new_node = TextNode(
    text      = "New fact added post-snapshot.",
    id_       = "new-node-x",
    embedding = embed_model.get_text_embedding("New fact added post-snapshot."),
)
vector_store.add([new_node])

h2 = vector_store.get_state_hash()
print(f"After new insert : {h2}")
print(f"Hashes differ    : {h1 != h2}")

## 11 · Snapshot & bit-exact restore

In [ ]:
hash_before = vector_store.get_state_hash()
snap        = vector_store.snapshot()

print(f"Snapshot size: {len(snap):,} bytes")
print(f"Hash before  : {hash_before}")

restored = ValoricoreLlamaIndex(path="/tmp/valori_llamaindex_restored")
restored.restore(snap)

hash_after = restored.get_state_hash()
print(f"Hash after   : {hash_after}")
print(f"Bit-exact    : {hash_before == hash_after} ✓")

## Summary

| What you did | Old way | With `valoricore.integrations` |
|---|---|---|
| Initialize | `ValoricoreAdapter(base_url=...) + LlamaIndexVectorStore(adapter=...)` | `ValoricoreLlamaIndex(path="./db")` |
| Insert nodes | manual `add_node` loop | `store.add(nodes)` |
| Query | `.search()` with manual score math | `store.query(VectorStoreQuery(...))` |
| Score range | broken (divided by 65536) | correct `(0, 1]` via `1/(1+L2²)` |
| Local embedded | not supported | `path="./db"` |
| State proof | not available | `store.get_state_hash()` |
| Snapshot | not available | `store.snapshot()` / `store.restore(snap)` |

**Next steps:**
- [LangChain Integration notebook](https://colab.research.google.com/drive/1HezK4l-Hbc6AdHxJNLwSqAgzr8WaKhiq)
- [End-to-End Demo notebook](https://colab.research.google.com/drive/1QO1yQMQoGbp9fwrb00KVKTq5bYVGXgJv)
- [GitHub: Valori-Kernel](https://github.com/varshith-Git/Valori-Kernel)